# Mexico City Temperature Analysis — 2023

In this notebook I'll explore monthly temperature data for Mexico as a country during 2025.
The data comes from **CONAGUA** (Comisión Nacional del Agua) — Mexico's national weather agency.

My goal is to use NumPy to uncover seasonal patterns, thermal behavior, and anomalies.
At this stage I am using no pandas, no matplotlib, just arrays and logic.

**Source:** https://smn.conagua.gob.mx/es/climatologia/temperaturas-y-lluvias/resumenes-mensuales-de-temperaturas-y-lluvias

---     

In [3]:
import numpy as np

The data was obtained from the official page of CONAGUA, and I'm going to use the national values, which are an average of temperatures from all the states.

In [4]:
# --- Data ---
months   = np.array(['Jan','Feb','Mar','Apr','May','Jun',
                     'Jul','Aug','Sep','Oct','Nov','Dec'])

temp_avg = np.array([15.9, 19.3, 20.8, 22.8, 25.5,26.4,
                     26.1, 26.6, 25.5, 23.5, 20.5, 18.8])

temp_max = np.array([23.7, 28.0, 29.7, 31.4, 33.5, 32.9,
                     32.2, 33.0, 31.4, 30.3, 28.4, 26.3])

temp_min = np.array([8.2,  10.6,  12.0, 14.1, 17.5, 20.0,
                     19.9, 20.1, 19.6, 16.7,  12.7,  11.4])
                     
                     

---
## Part 1 — Building the Temperature Matrix

Let's start by organizing all three temperature arrays into a single 2D structure.  
I'll stack `temp_avg`, `temp_max` and `temp_min` into a matrix where each row represents one metric and each column one month.

In [5]:
# Stack the three arrays into a (3, 12) matrix
# Row 0 = avg, Row 1 = max, Row 2 = min
temp_matrix = np.array([temp_avg,temp_max,temp_min]) 
print("Temperatures Matix:", temp_matrix)

Temperatures Matix: [[15.9 19.3 20.8 22.8 25.5 26.4 26.1 26.6 25.5 23.5 20.5 18.8]
 [23.7 28.  29.7 31.4 33.5 32.9 32.2 33.  31.4 30.3 28.4 26.3]
 [ 8.2 10.6 12.  14.1 17.5 20.  19.9 20.1 19.6 16.7 12.7 11.4]]


Now I'll start calculating some metrics

In [6]:
# - Calculate the mean of each metric across all months
mean_metric = temp_matrix.mean(axis = 1)
print("Mean temperature per metric", mean_metric)

# - Calculate the mean of each month across all metrics 
mean_month = temp_matrix.mean(axis = 0)
print("Mean temperature per month:", mean_month)

# What does each result tell us?
#The results shows the average temperature per metric and per months during 2025


Mean temperature per metric [22.64166667 30.06666667 15.23333333]
Mean temperature per month: [15.93333333 19.3        20.83333333 22.76666667 25.5        26.43333333
 26.06666667 26.56666667 25.5        23.5        20.53333333 18.83333333]


---
## Part 2 — Annual Overview

Now I'm going to get the big picture: hottest, coldest, average, and how much the temperature varies throughout the year.

In [58]:
# Let's find:
# - The hottest and coldest month (name + value)

print(f"Average hottest month: {months[temp_avg.argmax()]}, {temp_avg.max()} °C " )
print(f"Average coldest month: {months[temp_avg.argmin()]}, {temp_avg.min()} °C " )

# - The annual average temperature
annual_avg = temp_avg.mean()
print(f"Average anual temperature: {annual_avg:2f} °C " )

# - The temperature range (max avg - min avg)
print(f"Temperature range: {(temp_avg.max() - temp_avg.min()):2f} °C " )

# - How many degrees above/below the annual average each month is
print(f"Dregrees above/below average per month: {np.round((temp_avg-annual_avg),2)}")


Average hottest month: Aug, 26.6 °C 
Average coldest month: Jan, 15.9 °C 
Average anual temperature: 22.641667 °C 
Temperature range: 10.700000 °C 
Dregrees above/below average per month: [-6.74 -3.34 -1.84  0.16  2.86  3.76  3.46  3.96  2.86  0.86 -2.14 -3.84]


---
## Part 3 — Thermal Amplitude

Thermal amplitude is the difference between the daily max and min temperature.  
A high amplitude means cold nights and hot days — common in high-altitude cities like CDMX.

In [ ]:
# Calculate the thermal amplitude for each month (temp_max - temp_min)
thermal_amplitude = temp_max - temp_min
print(f"Termal amplitude per month: {thermal_amplitude} ")

# Then find: which month has the highest amplitude? Which has the lowest?
print(f"Month with highest amplitude: {months[thermal_amplitude.argmax()]}")
print(f"Month with lowest amplitude: {months[thermal_amplitude.argmin()]}")

# Print each month with its amplitude 
thermal_months = []
for m,a in zip(months,thermal_amplitude):
    thermal_months.append(f"{m} : {a}")
print(f"Amplitude per months: {thermal_months}")

Termal amplitude per month: [15.5 17.4 17.7 17.3 16.  12.9 12.3 12.9 11.8 13.6 15.7 14.9] 
Month with highest amplitude: Mar
Month with lowest amplitude: Sep
Amplitude per months: ['Jan : 15.5', 'Feb : 17.4', 'Mar : 17.7', 'Apr : 17.299999999999997', 'May : 16.0', 'Jun : 12.899999999999999', 'Jul : 12.300000000000004', 'Aug : 12.899999999999999', 'Sep : 11.799999999999997', 'Oct : 13.600000000000001', 'Nov : 15.7', 'Dec : 14.9']


---
## Part 4 — Seasonal Breakdown

Mexico City has distinct seasons that don't map to the traditional four.  
Let's use array indexing to group months and compare them.

In [24]:
# Season definitions for CDMX:
# Dry-cool  → Jan, Feb, Dec       (indices 0, 1, 11)
dry_cool_season = temp_avg[[0,1,11]]

# Dry-warm  → Mar, Apr, May       (indices 2, 3, 4)
dry_warm_season = temp_avg[2:5]

# Rainy     → Jun, Jul, Aug, Sep, Oct  (indices 5, 6, 7, 8, 9)
rainy_season = temp_avg[5:10]

# Transition → Nov                (index 10)
transition = temp_avg[10]

# Let's calculate the average temperature per season
season_names = np.array(['Dry_cool_season', 'Dry_warm_season', 'Rainy_season', 'Transition'])
season_avgs = np.array([dry_cool_season.mean(), dry_warm_season.mean(), rainy_season.mean(), transition.mean()])

for n,a in zip(season_names, season_avgs):
    print(f"{n}: {a:.2f}")

# Stack all season averages into one array and find the warmest and coolest season
print(f"Warmest season: {season_names[season_avgs.argmax()]}")
print(f"Coldest season: {season_names[season_avgs.argmin()]}")

Dry_cool_season: 18.00
Dry_warm_season: 23.03
Rainy_season: 25.62
Transition: 20.50
Warmest season: Rainy_season
Coldest season: Dry_cool_season


---
## Part 5 — Boolean Filtering

Now let's use conditional selection to ask specific questions about the data.

In [ ]:
# Which months are above the annual average temperature?
print(f"Annual Average: {annual_avg}")
print(f"Months above average: {months[temp_avg > annual_avg]}")
print(f"Months above average: {months[temp_avg <= annual_avg]}")



Annual Average: 22.641666666666666
Months above average: ['Apr' 'May' 'Jun' 'Jul' 'Aug' 'Sep' 'Oct']
Months above average: ['Jan' 'Feb' 'Mar' 'Nov' 'Dec']


In [36]:
# Let's go further: which months have BOTH high avg temp (> 17°C)
# AND high thermal amplitude (> 13°C)?
# This multi-condition filter tells us which months are warm but with large day/night swings

conditions = (temp_avg > 17) & (thermal_amplitude > 13)
print(f"Months with both high avg temp and high thermal amplitude: {months[conditions]}")

Months with both high avg temp and high thermal amplitude: ['Feb' 'Mar' 'Apr' 'May' 'Oct' 'Nov' 'Dec']


---
## Part 6 — Going Further

A few extra questions to push the analysis deeper.

In [47]:
# Normalize temp_avg to a 0-1 scale using:
# normalized = (x - min) / (max - min)
normalized_temp = (temp_avg - temp_avg.min()) / (temp_avg.max() - temp_avg.min())
print(f"Normalized temperatures: {normalized_temp.round(3)}")
# This is called Min-Max normalization — used constantly in ML preprocessing


# Which month has the highest normalized value? (should be the same as argmax, let's verify)
print(f"Month with highest normalized value: {months[normalized_temp.argmax()]}")


Normalized temperatures: [0.    0.318 0.458 0.645 0.897 0.981 0.953 1.    0.897 0.71  0.43  0.271]
Month with highest normalized value: Aug


In [56]:
# Calculate the month-over-month temperature change using np.diff()
month_change = np.diff(temp_avg)
print(f"Month over month change: {month_change}")

# Which transition has the biggest jump up? Which has the biggest drop?
print(f"Biggest jump up: {months[month_change.argmax()]} to {months[month_change.argmax() + 1]} : {month_change.max():.2f}°C ")
print(f"Biggest drop: {months[month_change.argmin()]} to {months[month_change.argmin() + 1]} : {month_change.min():.2f}°C ")

# Note: np.diff() returns an array of n-1 elements


Month over month change: [ 3.4  1.5  2.   2.7  0.9 -0.3  0.5 -1.1 -2.  -3.  -1.7]
Biggest jump up: Jan to Feb : 3.40°C 
Biggest drop: Oct to Nov : -3.00°C 


---
## Observations

*Write your findings here after completing the analysis.*

-   August is the hottest month, while January is the coldest
-  March is the month with the highest amplitude, which means is the month where the days are hot and the nights are cold, on the other hand, September is the one with the lowest amplitude, meaning the temperature does not chanche that radically
- The big seasonal changes occurr on January and October